# Preprocessing

In [1]:
import pandas as pd


In [2]:
df = pd.read_csv("SMSSpamCollection", sep="\t", header=None, names=['label', 'messages'])

# Check first few rows
print(df.head())

  label                                           messages
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [3]:
df['label'] = df['label'].map({'spam': 1, 'ham': 0})
print(df.head())

   label                                           messages
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


In [12]:
import string
import re

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['messages'] = df['messages'].apply(remove_punctuation).str.lower()

print(df.head())

   label                                           messages
0      0  go until jurong point crazy available only in ...
1      0                            ok lar joking wif u oni
2      1  free entry in 2 a wkly comp to win fa cup fina...
3      0        u dun say so early hor u c already then say
4      0  nah i dont think he goes to usf he lives aroun...


In below steps of votings:

*   We import `TfidfVectorizer` from `sklearn.feature_extraction.text`.
*   An instance of `TfidfVectorizer` is created. `max_features=5000` is used to limit the vocabulary size to the 5000 most frequent words, which helps manage computational complexity.
*   `tfidf.fit_transform(df['messages'])` calculates the TF-IDF scores for each word in each message and transforms the text data into a sparse matrix. `.toarray()` converts it to a dense NumPy array.
*   The resulting TF-IDF matrix is stored in `X`.
*   The `label` column (spam/ham) is extracted into `y`.
*   Finally, we print the shapes of `X` and `y` to confirm their dimensions, and show the first 10 learned feature names (words) from the TF-IDF vectorizer.

# Hard Voting

In [22]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import VotingClassifier

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

models = {
    "NaiveBayes": MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear', probability=True)
}

voting = VotingClassifier(
    estimators=[
        ('nb', MultinomialNB()),
        ('lr', LogisticRegression(max_iter=1000)),
        ('svm', SVC(kernel='linear', probability=True))
    ],
    voting='hard'
)

metrics = {
    name: {
        "precision": [],
        "recall": [],
        "f1": [],
        "confusion_matrix": [],
        "roc_auc": []
      }
    for name in list(models.keys()) + ["Voting"]
    }

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 42)

for train_idx, test_idx in skf.split(df['messages'], df['label']):

    X_train_text = df.iloc[train_idx]['messages']
    X_test_text = df.iloc[test_idx]['messages']

    y_train = df.iloc[train_idx]['label']
    y_test = df.iloc[test_idx]['label']

    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2) )
    # Limiting to 5000 features for manageable size

    X_train = tfidf.fit_transform(X_train_text)
    X_test = tfidf.transform(X_test_text)

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:,1] if hasattr(model, "predict_proba") else None

        # Collect metrics
        metrics[name]["precision"].append(precision_score(y_test, y_pred))
        metrics[name]["recall"].append(recall_score(y_test, y_pred))
        metrics[name]["f1"].append(f1_score(y_test, y_pred))
        metrics[name]["confusion_matrix"].append(confusion_matrix(y_test, y_pred))
        if y_prob is not None:
            metrics[name]["roc_auc"].append(roc_auc_score(y_test, y_prob))

    # Evaluate voting classifier
    voting.fit(X_train, y_train)
    y_pred_v = voting.predict(X_test)
    # For ROC-AUC, only available if voting='soft'
    y_prob_v = voting.predict_proba(X_test)[:,1] if hasattr(voting, "predict_proba") else None

    metrics["Voting"]["precision"].append(precision_score(y_test, y_pred_v))
    metrics["Voting"]["recall"].append(recall_score(y_test, y_pred_v))
    metrics["Voting"]["f1"].append(f1_score(y_test, y_pred_v))
    metrics["Voting"]["confusion_matrix"].append(confusion_matrix(y_test, y_pred_v))
    if y_prob_v is not None:
        metrics["Voting"]["roc_auc"].append(roc_auc_score(y_test, y_prob_v))




In [23]:
for model_name, model_metrics in metrics.items():
    print(f"\n=== Results for {model_name} ===")
    print(f"Precision scores: {model_metrics['precision']}")
    print(f"Recall scores: {model_metrics['recall']}")
    print(f"F1 scores: {model_metrics['f1']}")
    print(f"Confusion Matrices: {model_metrics['confusion_matrix']}")
    print(f"ROC-AUC scores: {model_metrics['roc_auc']}")

    # Print averages across folds
    print("\nAverages across folds:")
    if model_metrics['precision']:
        print(f"Avg Precision: {sum(model_metrics['precision'])/len(model_metrics['precision']):.4f}")
    if model_metrics['recall']:
        print(f"Avg Recall: {sum(model_metrics['recall'])/len(model_metrics['recall']):.4f}")
    if model_metrics['f1']:
        print(f"Avg F1: {sum(model_metrics['f1'])/len(model_metrics['f1']):.4f}")
    if model_metrics['roc_auc']:  # only if ROC-AUC was calculated
        print(f"Avg ROC-AUC: {sum(model_metrics['roc_auc'])/len(model_metrics['roc_auc']):.4f}")



=== Results for NaiveBayes ===
Precision scores: [1.0, 1.0, 1.0, 1.0, 1.0]
Recall scores: [0.8, 0.8, 0.7583892617449665, 0.7248322147651006, 0.7718120805369127]
F1 scores: [0.8888888888888888, 0.8888888888888888, 0.8625954198473282, 0.8404669260700389, 0.8712121212121212]
Confusion Matrices: [array([[965,   0],
       [ 30, 120]]), array([[965,   0],
       [ 30, 120]]), array([[965,   0],
       [ 36, 113]]), array([[965,   0],
       [ 41, 108]]), array([[965,   0],
       [ 34, 115]])]
ROC-AUC scores: [np.float64(0.988062176165803), np.float64(0.9870224525043177), np.float64(0.9837535208818722), np.float64(0.968619814306082), np.float64(0.9841986298988072)]

Averages across folds:
Avg Precision: 1.0000
Avg Recall: 0.7710
Avg F1: 0.8704
Avg ROC-AUC: 0.9823

=== Results for LogisticRegression ===
Precision scores: [0.9915966386554622, 0.9838709677419355, 0.991304347826087, 1.0, 1.0]
Recall scores: [0.7866666666666666, 0.8133333333333334, 0.7651006711409396, 0.7181208053691275, 0.7986

Hard voting reduced recall compared to SVM because weaker models (Naive Bayes and Logistic Regression) dominated the majority decision, overriding correct predictions made by SVM.

# Soft Voting

In [24]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import VotingClassifier, StackingClassifier

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

models = {
    "NaiveBayes": MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear', probability=True)
}

voting = VotingClassifier(
    estimators=[
        ('nb', MultinomialNB()),
        ('lr', LogisticRegression(max_iter=1000)),
        ('svm', SVC(kernel='linear', probability=True))
    ],
    voting='soft'
)

metrics = {
    name: {
        "precision": [],
        "recall": [],
        "f1": [],
        "confusion_matrix": [],
        "roc_auc": []
      }
    for name in list(models.keys()) + ["Voting"]
    }

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 42)

for train_idx, test_idx in skf.split(df['messages'], df['label']):

    X_train_text = df.iloc[train_idx]['messages']
    X_test_text = df.iloc[test_idx]['messages']

    y_train = df.iloc[train_idx]['label']
    y_test = df.iloc[test_idx]['label']

    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2) )
    # Limiting to 5000 features for manageable size

    X_train = tfidf.fit_transform(X_train_text)
    X_test = tfidf.transform(X_test_text)

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:,1] if hasattr(model, "predict_proba") else None

        # Collect metrics
        metrics[name]["precision"].append(precision_score(y_test, y_pred))
        metrics[name]["recall"].append(recall_score(y_test, y_pred))
        metrics[name]["f1"].append(f1_score(y_test, y_pred))
        metrics[name]["confusion_matrix"].append(confusion_matrix(y_test, y_pred))
        if y_prob is not None:
            metrics[name]["roc_auc"].append(roc_auc_score(y_test, y_prob))

    # Evaluate voting classifier
    voting.fit(X_train, y_train)
    y_pred_v = voting.predict(X_test)
    # For ROC-AUC, only available if voting='soft'
    y_prob_v = voting.predict_proba(X_test)[:,1] if hasattr(voting, "predict_proba") else None

    metrics["Voting"]["precision"].append(precision_score(y_test, y_pred_v))
    metrics["Voting"]["recall"].append(recall_score(y_test, y_pred_v))
    metrics["Voting"]["f1"].append(f1_score(y_test, y_pred_v))
    metrics["Voting"]["confusion_matrix"].append(confusion_matrix(y_test, y_pred_v))
    if y_prob_v is not None:
        metrics["Voting"]["roc_auc"].append(roc_auc_score(y_test, y_prob_v))




In [25]:
for model_name, model_metrics in metrics.items():
    print(f"\n=== Results for {model_name} ===")
    print(f"Precision scores: {model_metrics['precision']}")
    print(f"Recall scores: {model_metrics['recall']}")
    print(f"F1 scores: {model_metrics['f1']}")
    print(f"Confusion Matrices: {model_metrics['confusion_matrix']}")
    print(f"ROC-AUC scores: {model_metrics['roc_auc']}")

    # Print averages across folds
    print("\nAverages across folds:")
    if model_metrics['precision']:
        print(f"Avg Precision: {sum(model_metrics['precision'])/len(model_metrics['precision']):.4f}")
    if model_metrics['recall']:
        print(f"Avg Recall: {sum(model_metrics['recall'])/len(model_metrics['recall']):.4f}")
    if model_metrics['f1']:
        print(f"Avg F1: {sum(model_metrics['f1'])/len(model_metrics['f1']):.4f}")
    if model_metrics['roc_auc']:  # only if ROC-AUC was calculated
        print(f"Avg ROC-AUC: {sum(model_metrics['roc_auc'])/len(model_metrics['roc_auc']):.4f}")



=== Results for NaiveBayes ===
Precision scores: [1.0, 1.0, 1.0, 1.0, 1.0]
Recall scores: [0.8, 0.8, 0.7583892617449665, 0.7248322147651006, 0.7718120805369127]
F1 scores: [0.8888888888888888, 0.8888888888888888, 0.8625954198473282, 0.8404669260700389, 0.8712121212121212]
Confusion Matrices: [array([[965,   0],
       [ 30, 120]]), array([[965,   0],
       [ 30, 120]]), array([[965,   0],
       [ 36, 113]]), array([[965,   0],
       [ 41, 108]]), array([[965,   0],
       [ 34, 115]])]
ROC-AUC scores: [np.float64(0.988062176165803), np.float64(0.9870224525043177), np.float64(0.9837535208818722), np.float64(0.968619814306082), np.float64(0.9841986298988072)]

Averages across folds:
Avg Precision: 1.0000
Avg Recall: 0.7710
Avg F1: 0.8704
Avg ROC-AUC: 0.9823

=== Results for LogisticRegression ===
Precision scores: [0.9915966386554622, 0.9838709677419355, 0.991304347826087, 1.0, 1.0]
Recall scores: [0.7866666666666666, 0.8133333333333334, 0.7651006711409396, 0.7181208053691275, 0.7986

| Model           | Precision | Recall       | F1           | ROC-AUC    |
| --------------- | --------- | ------------ | ------------ | ---------- |
| Naive Bayes     | 1.000     | 0.771        | 0.870        | 0.982      |
| Logistic Reg    | 0.993     | 0.776        | 0.871        | 0.990      |
| **SVM**         | 0.993     | **0.902** 🔥 | **0.945** 🔥 | **0.9916** |
| Hard Voting     | 0.997     | 0.822        | 0.900        | ❌          |
| **Soft Voting** | 0.997     | 0.873        | 0.930        | 0.991      |


Soft voting improved performance over hard voting by incorporating prediction confidence, leading to better recall and F1-score. However, it still did not outperform the SVM model, which demonstrated superior ability to capture complex decision boundaries in high-dimensional text data.

The Multinomial Naive Bayes model achieved perfect precision, indicating no false positives. However, its recall was lower, meaning some spam messages were misclassified as ham. This suggests the model is conservative in labeling spam.

# Stack Classifier

In [26]:
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import VotingClassifier, StackingClassifier

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Base models
models = {
    "NaiveBayes": MultinomialNB(),
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear', probability=True)
}

# Voting classifier
voting = VotingClassifier(
    estimators=[
        ('nb', models["NaiveBayes"]),
        ('lr', models["LogisticRegression"]),
        ('svm', models["SVM"])
    ],
    voting='soft'
)

# Stacking classifier with Logistic Regression as final estimator
stacking = StackingClassifier(
    estimators=[
        ('nb', models["NaiveBayes"]),
        ('lr', models["LogisticRegression"]),
        ('svm', models["SVM"])
    ],
    final_estimator=LogisticRegression(max_iter=1000),
    stack_method='predict_proba'
)

# Metrics dictionary includes Voting and Stacking
metrics = {
    name: {"precision": [], "recall": [], "f1": [], "confusion_matrix": [], "roc_auc": []}
    for name in list(models.keys()) + ["Voting", "Stacking"]
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, test_idx in skf.split(df['messages'], df['label']):
    X_train_text = df.iloc[train_idx]['messages']
    X_test_text = df.iloc[test_idx]['messages']
    y_train = df.iloc[train_idx]['label']
    y_test = df.iloc[test_idx]['label']

    tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
    X_train = tfidf.fit_transform(X_train_text)
    X_test = tfidf.transform(X_test_text)

    # Evaluate individual models
    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:,1] if hasattr(model, "predict_proba") else None

        metrics[name]["precision"].append(precision_score(y_test, y_pred))
        metrics[name]["recall"].append(recall_score(y_test, y_pred))
        metrics[name]["f1"].append(f1_score(y_test, y_pred))
        metrics[name]["confusion_matrix"].append(confusion_matrix(y_test, y_pred))
        if y_prob is not None:
            metrics[name]["roc_auc"].append(roc_auc_score(y_test, y_prob))

    # Evaluate voting classifier
    voting.fit(X_train, y_train)
    y_pred_v = voting.predict(X_test)
    y_prob_v = voting.predict_proba(X_test)[:,1] if hasattr(voting, "predict_proba") else None

    metrics["Voting"]["precision"].append(precision_score(y_test, y_pred_v))
    metrics["Voting"]["recall"].append(recall_score(y_test, y_pred_v))
    metrics["Voting"]["f1"].append(f1_score(y_test, y_pred_v))
    metrics["Voting"]["confusion_matrix"].append(confusion_matrix(y_test, y_pred_v))
    if y_prob_v is not None:
        metrics["Voting"]["roc_auc"].append(roc_auc_score(y_test, y_prob_v))

    # Evaluate stacking classifier
    stacking.fit(X_train, y_train)
    y_pred_s = stacking.predict(X_test)
    y_prob_s = stacking.predict_proba(X_test)[:,1] if hasattr(stacking, "predict_proba") else None

    metrics["Stacking"]["precision"].append(precision_score(y_test, y_pred_s))
    metrics["Stacking"]["recall"].append(recall_score(y_test, y_pred_s))
    metrics["Stacking"]["f1"].append(f1_score(y_test, y_pred_s))
    metrics["Stacking"]["confusion_matrix"].append(confusion_matrix(y_test, y_pred_s))
    if y_prob_s is not None:
        metrics["Stacking"]["roc_auc"].append(roc_auc_score(y_test, y_prob_s))


In [27]:
for model_name, model_metrics in metrics.items():
    print(f"\n=== Results for {model_name} ===")
    print(f"Precision scores: {model_metrics['precision']}")
    print(f"Recall scores: {model_metrics['recall']}")
    print(f"F1 scores: {model_metrics['f1']}")
    print(f"Confusion Matrices: {model_metrics['confusion_matrix']}")
    print(f"ROC-AUC scores: {model_metrics['roc_auc']}")

    # Print averages across folds
    print("\nAverages across folds:")
    if model_metrics['precision']:
        print(f"Avg Precision: {sum(model_metrics['precision'])/len(model_metrics['precision']):.4f}")
    if model_metrics['recall']:
        print(f"Avg Recall: {sum(model_metrics['recall'])/len(model_metrics['recall']):.4f}")
    if model_metrics['f1']:
        print(f"Avg F1: {sum(model_metrics['f1'])/len(model_metrics['f1']):.4f}")
    if model_metrics['roc_auc']:  # only if ROC-AUC was calculated
        print(f"Avg ROC-AUC: {sum(model_metrics['roc_auc'])/len(model_metrics['roc_auc']):.4f}")



=== Results for NaiveBayes ===
Precision scores: [1.0, 1.0, 1.0, 1.0, 1.0]
Recall scores: [0.8, 0.8, 0.7583892617449665, 0.7248322147651006, 0.7718120805369127]
F1 scores: [0.8888888888888888, 0.8888888888888888, 0.8625954198473282, 0.8404669260700389, 0.8712121212121212]
Confusion Matrices: [array([[965,   0],
       [ 30, 120]]), array([[965,   0],
       [ 30, 120]]), array([[965,   0],
       [ 36, 113]]), array([[965,   0],
       [ 41, 108]]), array([[965,   0],
       [ 34, 115]])]
ROC-AUC scores: [np.float64(0.988062176165803), np.float64(0.9870224525043177), np.float64(0.9837535208818722), np.float64(0.968619814306082), np.float64(0.9841986298988072)]

Averages across folds:
Avg Precision: 1.0000
Avg Recall: 0.7710
Avg F1: 0.8704
Avg ROC-AUC: 0.9823

=== Results for LogisticRegression ===
Precision scores: [0.9915966386554622, 0.9838709677419355, 0.991304347826087, 1.0, 1.0]
Recall scores: [0.7866666666666666, 0.8133333333333334, 0.7651006711409396, 0.7181208053691275, 0.7986

# Observations


| Model        | Precision | Recall       | F1         | ROC-AUC    |
| ------------ | --------- | ------------ | ---------- | ---------- |
| Naive Bayes  | 1.000     | 0.771        | 0.870      | 0.982      |
| Logistic Reg | 0.993     | 0.776        | 0.871      | 0.990      |
| **SVM**      | 0.993     | 0.902        | **0.9451** | **0.9916** |
| Soft Voting  | 0.997     | 0.874        | 0.931      | 0.9908     |
| **Stacking** | 0.980     | **0.913** 🔥 | **0.9451** | 0.9908     |


Stacking outperformed voting by learning optimal combinations of base models using a meta-learner. It achieved the highest recall, indicating improved detection of spam messages, although with a slight decrease in precision due to increased false positives.

# Conclusion

Among all models, SVM and Stacking achieved the highest F1-scores, indicating the best balance between precision and recall. Stacking slightly outperformed SVM in recall, making it more suitable for applications where detecting spam is critical, even at the cost of a few false positives.